# 🖥️ ComfyUI en Google Colab — modo CPU

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/I-s-ax/Comfyui_en_colab/blob/main/comfyui_colab_cpu.ipynb)

Versión alternativa para ejecutar **ComfyUI sin GPU**, desde Android, usando la CPU disponible en Google Colab.

Incluye conexión con Google Drive, descarga de checkpoints/LoRA, lista de modelos e inicio de ComfyUI con un enlace web.

**Limitaciones:** la generación puede tardar mucho más que con una GPU. La generación de vídeo no es recomendable en CPU. Colab puede desconectar o limitar las sesiones; guardar modelos en Drive no guarda el estado de la sesión.

**Orden recomendado:** ejecuta las celdas de arriba hacia abajo. Si vienes de la versión GPU, detén la sesión anterior antes de iniciar esta para evitar dos procesos usando la misma carpeta de Drive.

## 1. Conectar Google Drive

In [ ]:
#@title 🔌 Conectar Google Drive
from google.colab import drive
from pathlib import Path

MONTAR_DRIVE = True #@param {type:"boolean"}
NOMBRE_CARPETA = "ComfyUI" #@param {type:"string"}

if MONTAR_DRIVE:
    drive.mount("/content/drive")
    WORKSPACE = str(Path("/content/drive/MyDrive") / NOMBRE_CARPETA)
else:
    WORKSPACE = "/content/ComfyUI"

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
print("Espacio de trabajo:", WORKSPACE)
print("Si usas la versión GPU, deja el mismo nombre de carpeta para reutilizar los modelos guardados en Drive.")

## 2. Instalar o actualizar ComfyUI
Esta celda usa la carpeta seleccionada arriba. Si ya existe una instalación Git, puede actualizarla; los modelos guardados en `models` no se borran.

In [ ]:
#@title 🧰 Instalar / actualizar ComfyUI para CPU
import subprocess, sys
from pathlib import Path

ACTUALIZAR_COMFYUI = True #@param {type:"boolean"}
repo = Path(WORKSPACE)

if (repo / ".git").exists():
    if ACTUALIZAR_COMFYUI:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
elif not (repo / "main.py").exists():
    if any(repo.iterdir()):
        raise RuntimeError(
            f"{repo} ya contiene archivos pero no parece una instalación de ComfyUI. "
            "Cambia NOMBRE_CARPETA en la primera celda o revisa la carpeta."
        )
    subprocess.run(
        ["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(repo)],
        check=True
    )

if not (repo / "main.py").exists():
    raise FileNotFoundError(f"No encuentro main.py dentro de {repo}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")],
    check=True
)

for name in [
    "checkpoints", "vae", "loras", "controlnet", "upscale_models",
    "diffusion_models", "text_encoders", "clip_vision"
]:
    (repo / "models" / name).mkdir(parents=True, exist_ok=True)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    print("ComfyUI se iniciará explícitamente con --cpu, aunque el entorno tenga CUDA instalada.")
except Exception as e:
    print("Aviso al comprobar PyTorch:", e)

print("ComfyUI preparado en:", repo)

## 3. Descargar un modelo
Pega el enlace directo de descarga de Civitai o de otro proveedor. Para Civitai puedes usar un token manual o guardarlo en Colab Secrets como `CIVITAI_API_TOKEN`. No compartas tu token.

In [ ]:
#@title 📥 Descargar checkpoint, LoRA u otro modelo
URL_MODELO = "" #@param {type:"string"}
CARPETA_DESTINO = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]
NOMBRE_ARCHIVO = "" #@param {type:"string"}
TOKEN_CIVITAI_MANUAL = "" #@param {type:"string"}

import os, re, time, requests
from pathlib import Path
from urllib.parse import urlparse, unquote

if not URL_MODELO.strip():
    raise ValueError("Pega el enlace de descarga en URL_MODELO.")

token = TOKEN_CIVITAI_MANUAL.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("CIVITAI_API_TOKEN") or ""
    except Exception:
        token = ""

original_url = URL_MODELO.strip()
url = original_url
if "civitai.com" in urlparse(url).netloc.lower() and token:
    url += ("&" if "?" in url else "?") + "token=" + token

dest = Path(WORKSPACE) / "models" / CARPETA_DESTINO
dest.mkdir(parents=True, exist_ok=True)
filename = NOMBRE_ARCHIVO.strip() or unquote(Path(urlparse(original_url).path).name)
if not filename or filename.lower() in ("download", "models"):
    filename = "modelo_descargado.safetensors"
target = dest / Path(filename).name
temp = target.with_name(target.name + ".part")

if target.exists() and target.stat().st_size:
    print(f"El archivo ya existe: {target} ({target.stat().st_size/1e9:.2f} GB)")
else:
    error = None
    for attempt in range(3):
        try:
            with requests.get(
                url, stream=True, allow_redirects=True, timeout=(30, 120),
                headers={"User-Agent": "ComfyUI-Colab-CPU/1.0"}
            ) as response:
                if response.status_code != 200:
                    raise RuntimeError(f"HTTP {response.status_code}; revisa el enlace y el token.")
                if "text/html" in response.headers.get("Content-Type", "").lower():
                    raise RuntimeError("El servidor devolvió HTML en vez del archivo del modelo.")
                total = int(response.headers.get("Content-Length", "0") or 0)
                got = 0
                with open(temp, "wb") as f:
                    for chunk in response.iter_content(8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            got += len(chunk)
                            if total:
                                print(f"Descargado {got/1e9:.2f}/{total/1e9:.2f} GB", end="\r")
                if total and got != total:
                    raise RuntimeError(f"Descarga incompleta: {got}/{total} bytes.")
                with open(temp, "rb") as f:
                    head = f.read(500).lower()
                if b"<html" in head or b"<!doctype" in head:
                    raise RuntimeError("El contenido parece HTML, no un modelo.")
                os.replace(temp, target)
                print(f"\nDescarga completada: {target} ({got/1e9:.2f} GB)")
                error = None
                break
        except Exception as e:
            error = e
            if temp.exists():
                temp.unlink()
            print(f"Intento {attempt + 1}/3: {e}")
            if attempt < 2:
                time.sleep(2 * (attempt + 1))
    if error:
        raise RuntimeError(f"No se pudo descargar el modelo: {error}") from error

print("Carpeta:", dest)

## 4. Listar los modelos descargados

In [ ]:
#@title 🔎 Listar archivos de modelos
MODELOS_A_LISTAR = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]

from pathlib import Path
folder = Path(WORKSPACE) / "models" / MODELOS_A_LISTAR
folder.mkdir(parents=True, exist_ok=True)
files = sorted(p for p in folder.iterdir() if p.is_file())
print("Carpeta:", folder)
if not files:
    print("No hay archivos en esta carpeta.")
for p in files:
    print(f"{p.name} — {p.stat().st_size / 1e9:.2f} GB")

## 5. Iniciar ComfyUI usando CPU
La opción `--cpu` fuerza el procesamiento por CPU. La primera carga puede tardar varios minutos. Cuando aparezca el enlace `https://....loca.lt`, ábrelo en otra pestaña y deja esta celda ejecutándose.

In [ ]:
#@title 🚀 Iniciar ComfyUI en CPU y mostrar enlace
import subprocess, time, re, sys, urllib.request, shutil, threading
from pathlib import Path

repo = Path(WORKSPACE)
if not (repo / "main.py").exists():
    raise FileNotFoundError("No encuentro main.py. Ejecuta primero la celda de instalación.")

def comprobar_comfyui():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188", timeout=3) as respuesta:
            return respuesta.status
    except Exception:
        return None

comfy = None
estado = comprobar_comfyui()
if estado:
    print(f"ComfyUI ya responde (HTTP {estado}); se reutilizará el proceso existente.", flush=True)
else:
    log_path = Path("/content/comfyui_cpu.log")
    log = open(log_path, "w", buffering=1)
    comfy = subprocess.Popen(
        [sys.executable, "-u", "main.py", "--cpu", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(repo), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    def leer_log():
        try:
            for linea in comfy.stdout:
                log.write(linea)
                print(linea, end="", flush=True)
        finally:
            log.flush()

    threading.Thread(target=leer_log, daemon=True).start()
    print("⏳ Iniciando ComfyUI en CPU; no detengas esta celda mientras carga.", flush=True)
    print("Registro: /content/comfyui_cpu.log", flush=True)
    inicio = time.time()
    limite = 600
    while time.time() - inicio < limite:
        estado = comprobar_comfyui()
        if estado:
            print(f"\n✅ ComfyUI responde correctamente (HTTP {estado}).", flush=True)
            break
        if comfy.poll() is not None:
            break
        time.sleep(3)

    log.close()
    if not estado:
        detalle = log_path.read_text(errors="ignore")[-6000:] if log_path.exists() else ""
        print("\n===== FINAL DEL REGISTRO =====\n", detalle, flush=True)
        raise RuntimeError(
            "ComfyUI no respondió en 10 minutos. Revisa el registro mostrado arriba. "
            "En CPU, la carga inicial puede ser lenta."
        )

# Preparar LocalTunnel, el método de enlace usado como primera opción.
if not shutil.which("npm"):
    raise RuntimeError("No se encontró npm en este entorno. Reinicia el entorno de Colab e inténtalo otra vez.")
if not shutil.which("lt"):
    instalacion = subprocess.run(
        ["npm", "install", "-g", "localtunnel"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(instalacion.stdout[-2500:], flush=True)
    if instalacion.returncode != 0 or not shutil.which("lt"):
        raise RuntimeError("No se pudo instalar LocalTunnel.")

print("\n🔑 Si LocalTunnel pide contraseña, intenta usar la dirección IP que muestra esta consulta:")
try:
    req = urllib.request.Request("https://loca.lt/mytunnelpassword", headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=15) as response:
        print(response.read().decode().strip(), flush=True)
except Exception as e:
    print("No se pudo consultar automáticamente la contraseña:", e, flush=True)

print("\n🌐 Creando enlace web. Espera una dirección https://....loca.lt", flush=True)
tunnel = subprocess.Popen(
    ["lt", "--port", "8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
url_encontrada = False
try:
    for linea in tunnel.stdout:
        print(linea, end="", flush=True)
        m = re.search(r"https://[A-Za-z0-9.-]+\.loca\.lt", linea)
        if m and not url_encontrada:
            url_encontrada = True
            print("\n✅ Abre este enlace en otra pestaña: " + m.group(0), flush=True)
            print("Deja esta celda ejecutándose mientras uses ComfyUI.", flush=True)
except KeyboardInterrupt:
    print("Se detuvo el enlace web.", flush=True)
finally:
    if tunnel.poll() is None:
        tunnel.terminate()
    if comfy is not None and comfy.poll() is None:
        print("ComfyUI sigue activo hasta que se cierre o se reinicie el entorno.", flush=True)

## Solución de problemas

- **La página tarda mucho en generar:** es normal en CPU. Prueba primero una resolución baja, por ejemplo 512 × 512, batch 1 y pocos pasos.
- **Error de memoria:** usa un checkpoint más pequeño y evita modelos grandes de vídeo.
- **El checkpoint no aparece:** revisa que esté en `ComfyUI/models/checkpoints`, que el archivo no termine en `.part` y refresca la interfaz.
- **Civitai devuelve 401/403:** revisa el enlace, los permisos y el token. No publiques tu token.
- **No aparece el enlace:** revisa los mensajes de LocalTunnel; si falla, vuelve a ejecutar la celda después de comprobar que ComfyUI esté activo.
- **Se desconecta Colab:** la sesión y el proceso pueden detenerse aunque los archivos guardados en Drive permanezcan.

El enlace de LocalTunnel es temporal. No compartas el enlace si tienes contenido privado abierto en la sesión.